In [5]:
import os
import time
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from sklearn.cluster import MeanShift
from sklearn.metrics import jaccard_score

# ==========================================
# 1. POINTNET++ UTILITIES (Pure PyTorch)
# ==========================================

def square_distance(src, dst):
    B, N, _ = src.shape
    _, M, _ = dst.shape
    dist = -2 * torch.matmul(src, dst.permute(0, 2, 1))
    dist += torch.sum(src ** 2, -1).view(B, N, 1)
    dist += torch.sum(dst ** 2, -1).view(B, 1, M)
    return dist

def index_points(points, idx):
    device = points.device
    B = points.shape[0]
    view_shape = list(idx.shape)
    view_shape[1:] = [1] * (len(view_shape) - 1)
    repeat_shape = list(idx.shape)
    repeat_shape[0] = 1
    batch_indices = torch.arange(B, dtype=torch.long).to(device).view(view_shape).repeat(repeat_shape)
    new_points = points[batch_indices, idx, :]
    return new_points

def farthest_point_sample(xyz, npoint):
    device = xyz.device
    B, N, C = xyz.shape
    centroids = torch.zeros(B, npoint, dtype=torch.long).to(device)
    distance = torch.ones(B, N).to(device) * 1e10
    farthest = torch.randint(0, N, (B,), dtype=torch.long).to(device)
    batch_indices = torch.arange(B, dtype=torch.long).to(device)
    for i in range(npoint):
        centroids[:, i] = farthest
        centroid = xyz[batch_indices, farthest, :].view(B, 1, 3)
        dist = torch.sum((xyz - centroid) ** 2, -1)
        mask = dist < distance
        distance[mask] = dist[mask]
        farthest = torch.max(distance, -1)[1]
    return centroids

def query_ball_point(radius, nsample, xyz, new_xyz):
    device = xyz.device
    B, N, C = xyz.shape
    _, S, _ = new_xyz.shape
    group_idx = torch.arange(N, dtype=torch.long).to(device).view(1, 1, N).repeat([B, S, 1])
    sqrdists = square_distance(new_xyz, xyz)
    group_idx[sqrdists > radius ** 2] = N
    group_idx = group_idx.sort(dim=-1)[0][:, :, :nsample]
    group_first = group_idx[:, :, 0].view(B, S, 1).repeat([1, 1, nsample])
    mask = group_idx == N
    group_idx[mask] = group_first[mask]
    return group_idx

# ==========================================
# 2. NETWORK MODULES
# ==========================================

class PointNetSetAbstraction(nn.Module):
    def __init__(self, npoint, radius, nsample, in_channel, mlp, group_all=False):
        super(PointNetSetAbstraction, self).__init__()
        self.npoint = npoint
        self.radius = radius
        self.nsample = nsample
        self.mlp_convs = nn.ModuleList()
        self.mlp_bns = nn.ModuleList()
        last_channel = in_channel
        for out_channel in mlp:
            self.mlp_convs.append(nn.Conv2d(last_channel, out_channel, 1))
            self.mlp_bns.append(nn.BatchNorm2d(out_channel))
            last_channel = out_channel
        self.group_all = group_all

    def forward(self, xyz, points):
        xyz = xyz.permute(0, 2, 1)
        if points is not None:
            points = points.permute(0, 2, 1)
        if self.group_all:
            new_xyz = torch.zeros(xyz.shape[0], 1, 3).to(xyz.device) 
            grouped_xyz = xyz.view(xyz.shape[0], 1, xyz.shape[1], 3)
            if points is not None:
                grouped_points = points.view(points.shape[0], 1, points.shape[1], -1)
                new_points = torch.cat([grouped_xyz, grouped_points], dim=-1)
            else:
                new_points = grouped_xyz
        else:
            new_xyz_idx = farthest_point_sample(xyz, self.npoint)
            new_xyz = index_points(xyz, new_xyz_idx)
            idx = query_ball_point(self.radius, self.nsample, xyz, new_xyz)
            grouped_xyz = index_points(xyz, idx)
            grouped_xyz_norm = grouped_xyz - new_xyz.view(xyz.shape[0], self.npoint, 1, 3)
            if points is not None:
                grouped_points = index_points(points, idx)
                new_points = torch.cat([grouped_xyz_norm, grouped_points], dim=-1)
            else:
                new_points = grouped_xyz_norm
    
        new_points = new_points.permute(0, 3, 2, 1) 
        for i, conv in enumerate(self.mlp_convs):
            bn = self.mlp_bns[i]
            new_points = F.relu(bn(conv(new_points)))
        new_points = torch.max(new_points, 2)[0]
        return new_xyz.permute(0, 2, 1), new_points

class PointNetFeaturePropagation(nn.Module):
    def __init__(self, in_channel, mlp):
        super(PointNetFeaturePropagation, self).__init__()
        self.mlp_convs = nn.ModuleList()
        self.mlp_bns = nn.ModuleList()
        last_channel = in_channel
        for out_channel in mlp:
            self.mlp_convs.append(nn.Conv1d(last_channel, out_channel, 1))
            self.mlp_bns.append(nn.BatchNorm1d(out_channel))
            last_channel = out_channel

    def forward(self, xyz1, xyz2, points1, points2):
        B, C, N = xyz1.shape
        _, _, M = xyz2.shape
        dists = square_distance(xyz1.permute(0, 2, 1), xyz2.permute(0, 2, 1))
        dists, idx = dists.sort(dim=-1)
        dists, idx = dists[:, :, :3], idx[:, :, :3]
        dist_recip = 1.0 / (dists + 1e-8)
        norm = torch.sum(dist_recip, dim=2, keepdim=True)
        weight = dist_recip / norm
        interpolated_points = torch.sum(index_points(points2.permute(0, 2, 1), idx) * weight.view(B, N, 3, 1), dim=2)
        interpolated_points = interpolated_points.permute(0, 2, 1)

        if points1 is not None:
            new_points = torch.cat([points1, interpolated_points], dim=1)
        else:
            new_points = interpolated_points

        for i, conv in enumerate(self.mlp_convs):
            bn = self.mlp_bns[i]
            new_points = F.relu(bn(conv(new_points)))
        return new_points

class InterpolateBlock(nn.Module):
    def forward(self, xyz_up, xyz_down, features_down):
        xyz_up = xyz_up.permute(0, 2, 1)
        xyz_down = xyz_down.permute(0, 2, 1)
        dists = square_distance(xyz_up, xyz_down)
        dists, idx = dists.sort(dim=-1)
        dists, idx = dists[:, :, :3], idx[:, :, :3]
        dist_recip = 1.0 / (dists + 1e-8)
        norm = torch.sum(dist_recip, dim=2, keepdim=True)
        weight = dist_recip / norm
        features_down = features_down.permute(0, 2, 1)
        interpolated = torch.sum(index_points(features_down, idx) * weight.view(features_down.shape[0], xyz_up.shape[1], 3, 1), dim=2)
        return interpolated.permute(0, 2, 1)

class PCFFModule(nn.Module):
    def __init__(self):
        super(PCFFModule, self).__init__()
        self.up = InterpolateBlock()
        self.conv1 = nn.Sequential(nn.Conv1d(256, 128, 1), nn.BatchNorm1d(128), nn.ReLU())
        self.conv2 = nn.Sequential(nn.Conv1d(128, 128, 1), nn.BatchNorm1d(128), nn.ReLU())

    def forward(self, xyz_a, xyz_b, xyz_c, f_na, f_nb, f_nc):
        f_nb_up = self.up(xyz_a, xyz_b, f_nb) 
        concat = torch.cat([f_na, f_nb_up], dim=1) 
        out = self.conv1(concat) 
        f_nc_up = self.up(xyz_a, xyz_c, f_nc) 
        out = out + f_nc_up 
        out = self.conv2(out) 
        return out
        
class JISSModule(nn.Module):
    def __init__(self, num_classes, emb_dim):
        super(JISSModule, self).__init__()
        self.ins_conv1 = nn.Sequential(nn.Conv1d(128, 128, 1), nn.BatchNorm1d(128), nn.ReLU())
        self.ins_conv2 = nn.Sequential(nn.Conv1d(256, 256, 1), nn.BatchNorm1d(256), nn.ReLU())
        self.ins_conv3 = nn.Sequential(nn.Conv1d(256, 128, 1), nn.BatchNorm1d(128), nn.ReLU())
        self.ins_out = nn.Conv1d(128, emb_dim, 1)
        
        self.sem_conv1 = nn.Sequential(nn.Conv1d(128, 128, 1), nn.BatchNorm1d(128), nn.ReLU())
        self.sem_conv2 = nn.Sequential(nn.Conv1d(256, 256, 1), nn.BatchNorm1d(256), nn.ReLU())
        self.sem_conv3 = nn.Sequential(nn.Conv1d(256, 128, 1), nn.BatchNorm1d(128), nn.ReLU())
        self.sem_out = nn.Conv1d(128, num_classes, 1)

    def forward(self, f_is, f_ss):
        B, C, N = f_is.shape
        sem_global = self.ins_conv1(f_ss)            
        sem_global = torch.mean(sem_global, dim=2, keepdim=True) 
        sem_tiled = sem_global.repeat(1, 1, N)       
        f_is_enhanced = f_is + sem_tiled            
        
        ins_concat = torch.cat([f_is, f_is_enhanced], dim=1) 
        ins_feat = self.ins_conv2(ins_concat)        
        ins_att = torch.mean(ins_feat, dim=1, keepdim=True) 
        ins_att = torch.sigmoid(ins_att)            
        ins_feat = ins_feat * ins_att                
        ins_feat = self.ins_conv3(ins_feat)          
        e_iss = self.ins_out(ins_feat)              
        
        ins_local = self.sem_conv1(f_is)            
        f_ss_enhanced = f_ss + ins_local            
        
        sem_concat = torch.cat([f_ss, f_ss_enhanced], dim=1) 
        sem_feat = self.sem_conv2(sem_concat)        
        sem_att = torch.mean(sem_feat, dim=1, keepdim=True) 
        sem_att = torch.sigmoid(sem_att)            
        sem_feat = sem_feat * sem_att                
        sem_feat = self.sem_conv3(sem_feat)          
        p_ssi = self.sem_out(sem_feat)              
        
        return p_ssi, e_iss

class ExactJSNet(nn.Module):
    def __init__(self, num_classes, emb_dim=5, in_channels=4):
        super(ExactJSNet, self).__init__()
        self.in_channels = in_channels
        
        # Optimized 4,096 Bottleneck Progression
        self.sa0 = PointNetSetAbstraction(npoint=4096, radius=0.4, nsample=32, in_channel=in_channels+3, mlp=[32, 32, 64], group_all=False)
        self.sa1 = PointNetSetAbstraction(npoint=1024, radius=0.8, nsample=32, in_channel=64+3, mlp=[64, 64, 128], group_all=False)
        self.sa2 = PointNetSetAbstraction(npoint=256,  radius=1.6, nsample=32, in_channel=128+3, mlp=[128, 128, 256], group_all=False)
        self.sa3 = PointNetSetAbstraction(npoint=64,   radius=3.2, nsample=32, in_channel=256+3, mlp=[256, 256, 512], group_all=False)
        self.sa4 = PointNetSetAbstraction(npoint=16,   radius=6.4, nsample=32, in_channel=512+3, mlp=[512, 512, 1024], group_all=False)
        
        self.ins_fp4 = PointNetFeaturePropagation(in_channel=1024+512, mlp=[512, 512]) 
        self.ins_fp3 = PointNetFeaturePropagation(in_channel=512+256,  mlp=[256, 256])
        self.ins_fp2 = PointNetFeaturePropagation(in_channel=256+128,  mlp=[256, 128])
        self.ins_fp1 = PointNetFeaturePropagation(in_channel=128+64,   mlp=[128, 128])
        self.ins_fp0 = PointNetFeaturePropagation(in_channel=128+in_channels, mlp=[128, 128]) 

        self.sem_fp4 = PointNetFeaturePropagation(in_channel=1024+512, mlp=[512, 512])
        self.sem_fp3 = PointNetFeaturePropagation(in_channel=512+256,  mlp=[256, 256])
        self.sem_fp2 = PointNetFeaturePropagation(in_channel=256+128,  mlp=[256, 128])
        self.sem_fp1 = PointNetFeaturePropagation(in_channel=128+64,   mlp=[128, 128])
        self.sem_fp0 = PointNetFeaturePropagation(in_channel=128+in_channels, mlp=[128, 128]) 

        self.ins_pcff = PCFFModule()
        self.sem_pcff = PCFFModule()
        self.jiss = JISSModule(num_classes, emb_dim)

    # Added explicit 'features' argument
    def forward(self, xyz, features):
        l0_xyz = xyz
        l0_points = features
        
        l1_xyz, l1_points = self.sa0(l0_xyz, l0_points)
        l2_xyz, l2_points = self.sa1(l1_xyz, l1_points)
        l3_xyz, l3_points = self.sa2(l2_xyz, l2_points)
        l4_xyz, l4_points = self.sa3(l3_xyz, l3_points)
        l5_xyz, l5_points = self.sa4(l4_xyz, l4_points)

        i_l4_points = self.ins_fp4(l4_xyz, l5_xyz, l4_points, l5_points)
        i_l3_points = self.ins_fp3(l3_xyz, l4_xyz, l3_points, i_l4_points)
        i_l2_points = self.ins_fp2(l2_xyz, l3_xyz, l2_points, i_l3_points)
        i_l1_points = self.ins_fp1(l1_xyz, l2_xyz, l1_points, i_l2_points)
        i_l0_points = self.ins_fp0(l0_xyz, l1_xyz, l0_points, i_l1_points)

        s_l4_points = self.sem_fp4(l4_xyz, l5_xyz, l4_points, l5_points)
        s_l3_points = self.sem_fp3(l3_xyz, l4_xyz, l3_points, s_l4_points)
        s_l2_points = self.sem_fp2(l2_xyz, l3_xyz, l2_points, s_l3_points)
        s_l1_points = self.sem_fp1(l1_xyz, l2_xyz, l1_points, s_l2_points)
        s_l0_points = self.sem_fp0(l0_xyz, l1_xyz, l0_points, s_l1_points)

        f_is = self.ins_pcff(l0_xyz, l1_xyz, l2_xyz, i_l0_points, i_l1_points, i_l2_points)
        f_ss = self.sem_pcff(l0_xyz, l1_xyz, l2_xyz, s_l0_points, s_l1_points, s_l2_points)

        sem_pred, ins_emb = self.jiss(f_is, f_ss)
        return sem_pred, ins_emb

# ==========================================
# 3. DATASET & LOSS FUNCTIONS
# ==========================================

class BridgeDataset(Dataset):
    def __init__(self, data_list, is_training=False):
        self.data = data_list
        self.is_training = is_training

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        points_data = item['points'].copy() # Keep all 7 channels!
        
        # SAFETY NET 1: Clean any NaNs or Infs present in raw input data
        points_data = np.nan_to_num(points_data, nan=0.0, posinf=1.0, neginf=-1.0)
        
        xyz = points_data[:, :3]
        normals = points_data[:, 3:6]
        z_rel = points_data[:, 6:]
    
        # 1. PER-CLOUD MIN-MAX NORMALIZATION FOR Z_REL
        # Scales relative height strictly to [0, 1] so height ratios remain identical across all bridges
        z_min = np.min(z_rel)
        z_max = np.max(z_rel)
        if z_max - z_min > 1e-6:
            z_rel = (z_rel - z_min) / (z_max - z_min)
        else:
            z_rel = np.zeros_like(z_rel)
        
        # SAFETY NET 2: Force z_rel strictly into [0, 1] bounds against floating-point precision drift
        z_rel = np.clip(z_rel, 0.0, 1.0)
    
        # Mean Centering (Only for XYZ)
        centroid = np.mean(xyz, axis=0)
        xyz -= centroid
        
        if self.is_training:
            # Z-axis Rotation (Normals must rotate too!)
            theta = np.random.uniform(0, 2 * np.pi)
            rot_mat = np.array([[np.cos(theta), -np.sin(theta), 0],
                                [np.sin(theta), np.cos(theta),  0],
                                [0,             0,              1]])
            xyz = xyz @ rot_mat
            normals = normals @ rot_mat
            
            # Anisotropic Scaling
            scale = np.random.uniform(0.9, 1.1, size=(3,))
            xyz = xyz * scale
            
            # Normal directions transform with the inverse-transpose of the scaling matrix
            scale_inv_T = np.diag(1.0 / scale)
            normals = normals @ scale_inv_T
            
            # Re-normalize normals
            norms = np.linalg.norm(normals, axis=1, keepdims=True)
            normals = normals / (norms + 1e-8)
            
            # Jittering (Apply only to XYZ coordinates)
            xyz += np.clip(np.random.normal(0, 0.01, size=xyz.shape), -0.05, 0.05)
    
        # SAFETY NET 3: Clamp surface normals strictly to valid unit range [-1.0, 1.0]
        normals = np.clip(normals, -1.0, 1.0)
    
        # Re-stack augmented data [XYZ, Normals, normalized z_rel]
        augmented_points = np.hstack((xyz, normals, z_rel))
    
        # SAFETY NET 4: Final verification to guarantee zero NaNs/Infs reach PyTorch CUDA tensors
        augmented_points = np.nan_to_num(augmented_points, nan=0.0, posinf=1.0, neginf=-1.0)
    
        return {
            'points': torch.tensor(augmented_points, dtype=torch.float32).transpose(1, 0),
            'semantic': torch.tensor(item['semantic'], dtype=torch.long),
            'instance': torch.tensor(item['instance'], dtype=torch.long)
        }

class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, alpha=None):
        super().__init__()
        self.gamma = gamma
        self.alpha = alpha 

    def forward(self, inputs, targets):
        ce_loss = F.cross_entropy(inputs, targets, weight=self.alpha, reduction='none')
        pt = torch.exp(-ce_loss)
        return (((1 - pt) ** self.gamma) * ce_loss).mean()

def discriminative_loss(embeddings, labels, delta_v=0.5, delta_d=1.5):
    batch_size, emb_dim, num_points = embeddings.size()
    l_var, l_dist, l_reg = 0.0, 0.0, 0.0
    
    for b in range(batch_size):
        emb_b = embeddings[b]
        lab_b = labels[b]
        unique_instances = torch.unique(lab_b)
        num_instances = len(unique_instances)
        
        mu = []
        l_var_b = 0.0
        for ins_id in unique_instances:
            ins_points = emb_b[:, lab_b == ins_id]
            mu_k = torch.mean(ins_points, dim=1, keepdim=True)
            mu.append(mu_k.squeeze())
            dist = torch.norm(ins_points - mu_k, p=2, dim=0)
            dist = torch.clamp(dist - delta_v, min=0.0)
            l_var_b += torch.mean(dist ** 2)
            
        l_var_b /= num_instances
        l_var += l_var_b
        
        l_dist_b = 0.0
        if num_instances > 1:
            mu_tensor = torch.stack(mu)
            for i in range(num_instances):
                for j in range(i + 1, num_instances):
                    dist_ij = torch.norm(mu_tensor[i] - mu_tensor[j], p=2)
                    dist_ij = torch.clamp(2 * delta_d - dist_ij, min=0.0)
                    l_dist_b += (dist_ij ** 2)
            l_dist_b /= (num_instances * (num_instances - 1))
        l_dist += l_dist_b
        l_reg += torch.mean(torch.norm(torch.stack(mu), p=2, dim=1))
        
    return (l_var + l_dist + 0.001 * l_reg) / batch_size

# ==========================================
# 4. EVALUATION & MAIN LOOP
# ==========================================

def evaluate_metrics(model, dataloader, num_classes=8):
    model.eval()
    all_sem_preds, all_sem_labels = [], []
    
    with torch.no_grad():
        for batch in dataloader:
            points = batch['points'].cuda()
            
            # Split features for updated forward pass
            xyz = points[:, :3, :].contiguous()
            features = points[:, 3:, :].contiguous()
            
            labels = batch['semantic'].cpu().numpy()
            sem_preds, _ = model(xyz, features)
            preds = sem_preds.argmax(dim=1).cpu().numpy()
            
            all_sem_preds.append(preds.flatten())
            all_sem_labels.append(labels.flatten())
            
    all_sem_preds = np.concatenate(all_sem_preds)
    all_sem_labels = np.concatenate(all_sem_labels)
    
    class_ious = jaccard_score(all_sem_labels, all_sem_preds, average=None, labels=np.arange(num_classes))
    miou = np.nanmean(class_ious)
    return miou, class_ious


def eval_semantic(pred, target, num_classes=8):
    """
    Literature-standard semantic metrics evaluation.
    Expects pred of shape [B, C, N] or [C, N].
    """
    if pred.dim() == 3:
        pred = pred.argmax(dim=1).cpu().numpy().flatten()
    else:
        pred = pred.argmax(dim=0).cpu().numpy().flatten()
        
    target = target.cpu().numpy().flatten()
    
    correct = (pred == target)
    oa = np.sum(correct) / len(target)
    
    ious, accs = [], []
    class_ious = np.zeros(num_classes)
    
    for c in range(num_classes):
        p_mask, t_mask = (pred == c), (target == c)
        intersection = np.sum(p_mask & t_mask)
        union = np.sum(p_mask | t_mask)
        gt_tot = np.sum(t_mask)
        
        if gt_tot > 0: 
            accs.append(intersection / gt_tot)
            
        if union > 0: 
            iou_val = intersection / union
            ious.append(iou_val)
            class_ious[c] = iou_val
            
    macc = np.mean(accs) if len(accs) > 0 else 0.0
    miou = np.mean(ious) if len(ious) > 0 else 0.0
    
    return oa, macc, miou, class_ious


def eval_instance(pred_emb, target, bandwidth=0.6, iou_thresh=0.5):
    """
    Literature-standard instance metrics evaluation using MeanShift clustering.
    Expects pred_emb of shape [E, N] and target of shape [N].
    """
    if isinstance(pred_emb, torch.Tensor):
        if pred_emb.dim() == 3:
            pred_emb = pred_emb[0]  # Remove batch dim -> [E, N]
        pred_emb_np = pred_emb.transpose(1, 0).detach().cpu().numpy() # -> [N, E]
    else:
        pred_emb_np = pred_emb.T

    if isinstance(target, torch.Tensor):
        target_np = target.detach().cpu().numpy().flatten()
    else:
        target_np = target.flatten()

    # 1. Cluster embeddings using MeanShift as per JSNet paper
    ms = MeanShift(bandwidth=bandwidth, bin_seeding=True)
    pred_ins = ms.fit_predict(pred_emb_np)

    gt_u, pred_u = np.unique(target_np), np.unique(pred_ins)
    cov, w_cov, tps, fns = 0.0, 0.0, 0, 0
    matched = set()

    for gt_id in gt_u:
        gt_mask = (target_np == gt_id)
        gt_sz = np.sum(gt_mask)
        max_iou, best_p = 0.0, -1
        
        for p_id in pred_u:
            p_mask = (pred_ins == p_id)
            union = np.sum(gt_mask | p_mask)
            iou = np.sum(gt_mask & p_mask) / (union + 1e-8)
            if iou > max_iou: 
                max_iou, best_p = iou, p_id

        cov += max_iou
        w_cov += (gt_sz * max_iou)
        
        if max_iou >= iou_thresh:
            tps += 1
            matched.add(best_p)
        else:
            fns += 1

    fps = len(pred_u) - len(matched)
    precision = tps / (tps + fps) if (tps + fps) > 0 else 0.0
    recall = tps / (tps + fns) if (tps + fns) > 0 else 0.0
    
    mean_cov = cov / len(gt_u) if len(gt_u) > 0 else 0.0
    mean_wcov = w_cov / len(target_np) if len(target_np) > 0 else 0.0
    
    return precision, recall, mean_cov, mean_wcov


def check_class_distribution(data_list, split_name):
    culvert_count = sum(1 for item in data_list if 6 in np.unique(item['semantic']))
    crossgirder_count = sum(1 for item in data_list if 3 in np.unique(item['semantic']))
    print(f"{split_name}: {culvert_count} / {len(data_list)} samples contain Culverts")
    print(f"{split_name}: {crossgirder_count} / {len(data_list)} samples contain cross-girders")

def verify_split(data_list, name):
    culv_count = sum(1 for item in data_list if 6 in np.unique(item['semantic']))
    cg_count   = sum(1 for item in data_list if 3 in np.unique(item['semantic']))
    print(f"{name} ({len(data_list)} total): {culv_count} Culverts | {cg_count} Cross-girders")
    

# Assuming your split and balancing code remains unchanged up to main()



In [14]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
base_dir = r"/storage/jothi/Yash/Datasets/Results_from_dataset_only_culverts_and_cross-girders_with_surfnormals"
log_file_path = os.path.join(base_dir, "training_log.txt")

train_data_list = np.load(f"{base_dir}/bridge_train_mod.npy", allow_pickle=True)
val_data_list   = np.load(f"{base_dir}/bridge_val_mod.npy", allow_pickle=True)
test_data_list  = np.load(f"{base_dir}/bridge_test_mod.npy", allow_pickle=True)

train_data_list[0]['points'][0]

array([-5.21121890e+00,  2.15518836e-01,  1.39696860e+00, -7.01483817e-04,
        1.82513683e-03,  9.99998093e-01,  4.35374689e+00])

In [16]:
## ====================================================
##   CODE FOR DATASET HAVING ONLY CROSS_GIRDER BRIDGES AND CULVERTS

import os
import time
import torch
import numpy as np
from torch.utils.data import DataLoader, WeightedRandomSampler

def main():

    EPOCHS = 120

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    base_dir = r"/storage/jothi/Yash/Datasets/Results_from_dataset_only_culverts_and_cross-girders_with_surfnormals"
    log_file_path = os.path.join(base_dir, "training_log.txt")

    train_data_list = np.load(f"{base_dir}/bridge_train_mod.npy", allow_pickle=True)
    val_data_list   = np.load(f"{base_dir}/bridge_val_mod.npy", allow_pickle=True)
    test_data_list  = np.load(f"{base_dir}/bridge_test_mod.npy", allow_pickle=True)

    # 1. WEIGHTED RANDOM SAMPLER FOR OVERSAMPLING SPARSE CLOUDS
    sample_weights = []
    for item in train_data_list:
        unique_classes = set(np.unique(item['semantic']))
        if 6 in unique_classes:          
            sample_weights.append(3.0)   # Reduced from 8.0
        else:                            
            sample_weights.append(1.0)   

    sampler = WeightedRandomSampler(
        weights=torch.DoubleTensor(sample_weights),
        num_samples=len(train_data_list),
        replacement=True
    )

    train_loader = DataLoader(BridgeDataset(train_data_list, True), batch_size=1, sampler=sampler)
    val_loader   = DataLoader(BridgeDataset(val_data_list, False), batch_size=1, shuffle=False)
    test_loader  = DataLoader(BridgeDataset(test_data_list, False), batch_size=1, shuffle=False)

    # UPDATED: model instantiated with in_channels=4 (3 for normals, 1 for z_rel)
    model = ExactJSNet(num_classes=8, emb_dim=5, in_channels=4).cuda()
    
    # 2. OPTIMIZER & SCHEDULER
    optimizer = torch.optim.AdamW(model.parameters(), lr=0.001, weight_decay=2e-3)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-5)
    
    # 3. UPDATED LOSS WEIGHTS FOR COMPONENT-LEVEL SPARSITY
    # Indices: [Railings, Deck, Girder, Cross-girder, Pier cap, Pier, Culvert, Bearings]
    # Cross-girder (3) boosted to 5.0 (tiny volume). 
    # Culvert (6) boosted to 6.0. 
    # Deck (1) reduced to 0.5 because it dominates every single cloud.
    weights = np.array([1.0, 0.5, 1.0, 2.5, 1.0, 1.0, 1.5, 0.1]) 
    
    class_weights = torch.FloatTensor(weights).to(device)
    sem_criterion = FocalLoss(gamma=2.0, alpha=class_weights)

    best_val_miou, patience_counter = 0.0, 0
    class_names = ['Railings', 'Deck', 'Girder', 'Cross-girder', 'Pier cap', 'Pier', 'Culvert', 'Bearings']
    num_batches = len(train_loader)

    # Open live log file
    with open(log_file_path, "w") as log_file:
        def log(msg):
            log_file.write(msg + "\n")
            log_file.flush()
            # print(msg) # Added print so you can see it in console too

        log(f"Running on: {device}")
        log(f"Starting Training ({num_batches} batches per epoch, {EPOCHS} Max Epochs)...")

        for epoch in range(1, EPOCHS + 1):
            model.train()
            total_loss = 0.0
            start_time = time.time()
            
            for batch_idx, batch in enumerate(train_loader, 1):
                batch_start = time.time()
                optimizer.zero_grad()
                
                points = batch['points'].cuda()
                sem_labels = batch['semantic'].cuda()
                inst_labels = batch['instance'].cuda()
                
                # UPDATED: Split points into spatial coordinates and extra features
                xyz = points[:, :3, :].contiguous()
                features = points[:, 3:, :].contiguous()
                
                sem_preds, inst_embeds = model(xyz, features)
                
                loss_sem = sem_criterion(sem_preds, sem_labels)
                loss_inst = discriminative_loss(inst_embeds, inst_labels)
                loss = loss_sem + loss_inst
                
                loss.backward()
                optimizer.step()
                total_loss += loss.item()
                batch_dur = time.time() - batch_start
                
                if batch_idx % 10 == 0 or batch_idx == num_batches:
                    log(f"Epoch [{epoch}/{EPOCHS}] | Batch [{batch_idx}/{num_batches}] | Loss: {loss.item():.4f} (Sem: {loss_sem.item():.4f}, Ins: {loss_inst.item():.4f}) | Time: {batch_dur:.2f}s")
                
            scheduler.step()
            epoch_dur = time.time() - start_time
            
            # Validation every 5 epochs
            if epoch % 5 == 0:
                log(f"\n--- Running Validation for Epoch {epoch} ---")
                val_miou, val_class_ious = evaluate_metrics(model, val_loader)
                log(f"Epoch {epoch}/{EPOCHS} [{epoch_dur:.1f}s] | Train Loss: {total_loss/num_batches:.4f} | Val mIoU: {val_miou:.4f}")
                
                for i, name in enumerate(class_names):
                    log(f"  - {name} IoU: {val_class_ious[i]:.4f}")
                
                if val_miou > best_val_miou:
                    best_val_miou = val_miou
                    patience_counter = 0
                    torch.save(model.state_dict(), f"{base_dir}/best_optimized_jsnet.pth")
                    log("   --> Best model saved.\n")
                else:
                    patience_counter += 1
                    log(f"   --> No improvement ({patience_counter}/10 patience)\n")
                    if patience_counter >= 10:
                        log("\nEarly stopping triggered due to no improvement in validation mIoU.")
                        break
            else:
                log(f"Epoch [{epoch}/{EPOCHS}] Completed in {epoch_dur:.1f}s | Avg Loss: {total_loss/num_batches:.4f}\n")

    # ==========================================
    # FINAL TEST EVALUATION (EXACT LITERATURE METRICS)
    # ==========================================
    print("\nStarting Final Test Evaluation...", flush=True)
    model.load_state_dict(torch.load(f"{base_dir}/best_optimized_jsnet.pth"))
    model.eval()

    mets = {
        'oa': [], 'macc': [], 'miou': [],
        'prec': [], 'rec': [], 'cov': [], 'wcov': []
    }
    class_ious_accumulator = np.zeros(8)

    with torch.no_grad():
        for batch in test_loader:
            points = batch['points'].cuda()
            sem_labels = batch['semantic'].cuda()
            inst_labels = batch['instance'].cuda()

            # UPDATED: Split points into spatial coordinates and extra features for inference
            xyz = points[:, :3, :].contiguous()
            features = points[:, 3:, :].contiguous()

            sem_preds, inst_embeds = model(xyz, features)

            # 1. Literature Semantic Evaluation
            oa, macc, miou, class_ious = eval_semantic(sem_preds[0], sem_labels[0], num_classes=8)
            mets['oa'].append(oa)
            mets['macc'].append(macc)
            mets['miou'].append(miou)
            class_ious_accumulator += class_ious

            # 2. Literature Instance Evaluation (MeanShift)
            prec, rec, cov, wcov = eval_instance(inst_embeds[0], inst_labels[0], bandwidth=0.6)
            mets['prec'].append(prec)
            mets['rec'].append(rec)
            mets['cov'].append(cov)
            mets['wcov'].append(wcov)

    avg_class_ious = class_ious_accumulator / len(test_loader)

    print(f"\n================ FINAL TEST RESULTS ================", flush=True)
    print(f"Semantic -> OA: {np.mean(mets['oa']):.4f} | mAcc: {np.mean(mets['macc']):.4f} | mIoU: {np.mean(mets['miou']):.4f}")
    print(f"Instance -> mPrec: {np.mean(mets['prec']):.4f} | mRec: {np.mean(mets['rec']):.4f} | Cov: {np.mean(mets['cov']):.4f} | WCov: {np.mean(mets['wcov']):.4f}")
    print("--------------------------------------------------")
    for i, name in enumerate(class_names):
        print(f"  {name} Test IoU: {avg_class_ious[i]:.4f}")
    print("==================================================")
                    
if __name__ == '__main__':
    main()


Starting Final Test Evaluation...

================ FINAL TEST RESULTS ================
Semantic -> OA: 0.7989 | mAcc: 0.6928 | mIoU: 0.5241
Instance -> mPrec: 0.2842 | mRec: 0.2519 | Cov: 0.3469 | WCov: 0.5920
--------------------------------------------------
  Railings Test IoU: 0.6940
  Deck Test IoU: 0.8022
  Girder Test IoU: 0.6103
  Cross-girder Test IoU: 0.0182
  Pier cap Test IoU: 0.5586
  Pier Test IoU: 0.7942
  Culvert Test IoU: 0.0000
  Bearings Test IoU: 0.0000


In [17]:
# BALANCING bridge datsets with rare elements like culverts and cross-girders in training, validation and testing datasets

base_dir = r"/storage/jothi/Yash/Datasets/Results_from_133_datasets_with_surfnormals_modified"

# 1. Load existing splits
train_data = list(np.load(f"{base_dir}/bridge_train.npy", allow_pickle=True))
val_data   = list(np.load(f"{base_dir}/bridge_val.npy", allow_pickle=True))
test_data  = list(np.load(f"{base_dir}/bridge_test.npy", allow_pickle=True))
all_datasets = train_data + val_data + test_data

print("datasets having culvert, and cross-girder classes before modification:")
check_class_distribution(train_data, "Train")
check_class_distribution(val_data, "Val")
check_class_distribution(test_data, "Test")

# 2. Categorize by rare classes
culvert_bridges = []
cross_girder_only = []
standard_bridges = []

for item in all_datasets:
    classes = set(np.unique(item['semantic']))
    if 6 in classes:
        culvert_bridges.append(item)
    elif 3 in classes:
        cross_girder_only.append(item)
    else:
        standard_bridges.append(item)

# 3. Shuffle deterministically
np.random.seed(42)
np.random.shuffle(culvert_bridges)
np.random.shuffle(cross_girder_only)
np.random.shuffle(standard_bridges)

# 4. Stratify Culverts (5 total -> 3 Train, 1 Val, 1 Test)
train_culv = culvert_bridges[:3]
val_culv   = [culvert_bridges[3]]
test_culv  = [culvert_bridges[4]]

# 5. Stratify Cross-girder only bridges (~70% Train, 15% Val, 15% Test)
n_val_cg = int(len(cross_girder_only) * 0.15)
n_test_cg = int(len(cross_girder_only) * 0.15)

val_cg   = cross_girder_only[:n_val_cg]
test_cg  = cross_girder_only[n_val_cg:n_val_cg + n_test_cg]
train_cg = cross_girder_only[n_val_cg + n_test_cg:]

# 6. Stratify remaining standard bridges
n_val_std = int(len(standard_bridges) * 0.15)
n_test_std = int(len(standard_bridges) * 0.15)

val_std   = standard_bridges[:n_val_std]
test_std  = standard_bridges[n_val_std:n_val_std + n_test_std]
train_std = standard_bridges[n_val_std + n_test_std:]

# 7. Merge and save updated splits
final_train = train_culv + train_cg + train_std
final_val   = val_culv + val_cg + val_std
final_test  = test_culv + test_cg + test_std

np.save(f"{base_dir}/bridge_train_mod.npy", final_train)
np.save(f"{base_dir}/bridge_val_mod.npy", final_val)
np.save(f"{base_dir}/bridge_test_mod.npy", final_test)

print("--- Stratified Split Completed ---")
print(f"Train Set ({len(final_train)}): {len(train_culv)} Culverts, {len(train_cg) + len(train_culv)} Cross-girders")
print(f"Val Set   ({len(final_val)}): {len(val_culv)} Culverts, {len(val_cg) + len(val_culv)} Cross-girders")
print(f"Test Set  ({len(final_test)}): {len(test_culv)} Culverts, {len(test_cg) + len(test_culv)} Cross-girders")



datasets having culvert, and cross-girder classes before modification:
Train: 4 / 93 samples contain Culverts
Train: 26 / 93 samples contain cross-girders
Val: 1 / 20 samples contain Culverts
Val: 4 / 20 samples contain cross-girders
Test: 0 / 20 samples contain Culverts
Test: 6 / 20 samples contain cross-girders
--- Stratified Split Completed ---
Train Set (95): 3 Culverts, 29 Cross-girders
Val Set   (19): 1 Culverts, 6 Cross-girders
Test Set  (19): 1 Culverts, 6 Cross-girders


In [18]:
train_data = np.load(f"{base_dir}/bridge_train_mod.npy", allow_pickle=True)
val_data   = np.load(f"{base_dir}/bridge_val_mod.npy", allow_pickle=True)
test_data  = np.load(f"{base_dir}/bridge_test_mod.npy", allow_pickle=True)

verify_split(train_data, "Train")
verify_split(val_data,   "Val")
verify_split(test_data,  "Test")

Train (95 total): 3 Culverts | 26 Cross-girders
Val (19 total): 1 Culverts | 5 Cross-girders
Test (19 total): 1 Culverts | 5 Cross-girders


In [19]:
## ====================================================
## CODE FOR ALL DATASETS

def main():
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    base_dir = r"/storage/jothi/Yash/Datasets/Results_from_133_datasets_with_surfnormals_modified"
    log_file_path = os.path.join(base_dir, "training_log.txt")

    train_data_list = np.load(f"{base_dir}/bridge_train_mod.npy", allow_pickle=True)
    val_data_list   = np.load(f"{base_dir}/bridge_val_mod.npy", allow_pickle=True)
    test_data_list  = np.load(f"{base_dir}/bridge_test_mod.npy", allow_pickle=True)

    # 1. WEIGHTED RANDOM SAMPLER FOR OVERSAMPLING SPARSE CLASSES
    sample_weights = []
    for item in train_data_list:
        unique_classes = set(np.unique(item['semantic']))
        if 6 in unique_classes:          # Culvert present
            sample_weights.append(10.0)   # 5x higher chance of selection
        elif 3 in unique_classes:        # Cross-girder present
            sample_weights.append(5.0)   # 2x higher chance of selection
        elif 7 in unique_classes:        # Bearings present
            sample_weights.append(5.0)   # Boosted for tiny hardware
        else:
            sample_weights.append(1.0)   # Standard selection

    sampler = WeightedRandomSampler(
        weights=torch.DoubleTensor(sample_weights),
        num_samples=len(train_data_list),
        replacement=True
    )

    # Note: shuffle must be False when using sampler
    train_loader = DataLoader(BridgeDataset(train_data_list, True), batch_size=1, sampler=sampler)
    val_loader   = DataLoader(BridgeDataset(val_data_list, False), batch_size=1, shuffle=False)
    test_loader  = DataLoader(BridgeDataset(test_data_list, False), batch_size=1, shuffle=False)

    # UPDATED: model instantiated with in_channels=4 (3 for normals, 1 for z_rel)
    model = ExactJSNet(num_classes=8, emb_dim=5, in_channels=4).cuda()
    
    optimizer = torch.optim.AdamW(model.parameters(), lr=0.001, weight_decay=1e-4)
    # Maintain a minimum learning rate floor so minority gradients remain active
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=120, eta_min=1e-4)
    
    # 2. UPDATED LOSS WEIGHTS
    # Indices: [Railings, Deck, Girder, Cross-girder, Pier cap, Pier, Culvert, Bearings]
    weights = np.array([3.0, 0.5, 1.5, 5.0, 2.0, 1.5, 6.0, 6.0]) 
    class_weights = torch.FloatTensor(weights).to(device)
    sem_criterion = FocalLoss(gamma=1.5, alpha=class_weights)

    best_val_miou, patience_counter = 0.0, 0
    class_names = ['Railings', 'Deck', 'Girder', 'Cross-girder', 'Pier cap', 'Pier', 'Culvert', 'Bearings']
    num_batches = len(train_loader)

    # Open live log file
    with open(log_file_path, "w") as log_file:
        def log(msg):
            log_file.write(msg + "\n")
            log_file.flush()
            # print(msg) # Added print so you can see it in console too

        log(f"Running on: {device}")
        log(f"Starting Training ({num_batches} batches per epoch, 120 Max Epochs)...")

        for epoch in range(1, 121):
            model.train()
            total_loss = 0.0
            start_time = time.time()
            
            for batch_idx, batch in enumerate(train_loader, 1):
                batch_start = time.time()
                optimizer.zero_grad()
                
                points = batch['points'].cuda()
                sem_labels = batch['semantic'].cuda()
                inst_labels = batch['instance'].cuda()
                
                # UPDATED: Split points into spatial coordinates and extra features
                xyz = points[:, :3, :].contiguous()
                features = points[:, 3:, :].contiguous()
                
                sem_preds, inst_embeds = model(xyz, features)
                
                loss_sem = sem_criterion(sem_preds, sem_labels)
                loss_inst = discriminative_loss(inst_embeds, inst_labels)
                loss = loss_sem + loss_inst
                
                loss.backward()
                optimizer.step()
                total_loss += loss.item()
                batch_dur = time.time() - batch_start
                
                if batch_idx % 10 == 0 or batch_idx == num_batches:
                    log(f"Epoch [{epoch}/120] | Batch [{batch_idx}/{num_batches}] | Loss: {loss.item():.4f} (Sem: {loss_sem.item():.4f}, Ins: {loss_inst.item():.4f}) | Time: {batch_dur:.2f}s")
                
            scheduler.step()
            epoch_dur = time.time() - start_time
            
            # Validation every 5 epochs
            if epoch % 5 == 0:
                log(f"\n--- Running Validation for Epoch {epoch} ---")
                val_miou, val_class_ious = evaluate_metrics(model, val_loader)
                log(f"Epoch {epoch}/120 [{epoch_dur:.1f}s] | Train Loss: {total_loss/num_batches:.4f} | Val mIoU: {val_miou:.4f}")
                
                for i, name in enumerate(class_names):
                    log(f"  - {name} IoU: {val_class_ious[i]:.4f}")
                
                if val_miou > best_val_miou:
                    best_val_miou = val_miou
                    patience_counter = 0
                    torch.save(model.state_dict(), f"{base_dir}/best_optimized_jsnet.pth")
                    log("   --> Best model saved.\n")
                else:
                    patience_counter += 1
                    log(f"   --> No improvement ({patience_counter}/10 patience)\n")
                    if patience_counter >= 10:
                        log("\nEarly stopping triggered due to no improvement in validation mIoU.")
                        break
            else:
                log(f"Epoch [{epoch}/120] Completed in {epoch_dur:.1f}s | Avg Loss: {total_loss/num_batches:.4f}\n")

    # ==========================================
    # FINAL TEST EVALUATION (EXACT LITERATURE METRICS)
    # ==========================================
    print("\nStarting Final Test Evaluation...", flush=True)
    model.load_state_dict(torch.load(f"{base_dir}/best_optimized_jsnet.pth"))
    model.eval()

    mets = {
        'oa': [], 'macc': [], 'miou': [],
        'prec': [], 'rec': [], 'cov': [], 'wcov': []
    }
    class_ious_accumulator = np.zeros(8)

    with torch.no_grad():
        for batch in test_loader:
            points = batch['points'].cuda()
            sem_labels = batch['semantic'].cuda()
            inst_labels = batch['instance'].cuda()

            # UPDATED: Split points into spatial coordinates and extra features for inference
            xyz = points[:, :3, :].contiguous()
            features = points[:, 3:, :].contiguous()

            sem_preds, inst_embeds = model(xyz, features)

            # 1. Literature Semantic Evaluation
            oa, macc, miou, class_ious = eval_semantic(sem_preds[0], sem_labels[0], num_classes=8)
            mets['oa'].append(oa)
            mets['macc'].append(macc)
            mets['miou'].append(miou)
            class_ious_accumulator += class_ious

            # 2. Literature Instance Evaluation (MeanShift)
            prec, rec, cov, wcov = eval_instance(inst_embeds[0], inst_labels[0], bandwidth=0.6)
            mets['prec'].append(prec)
            mets['rec'].append(rec)
            mets['cov'].append(cov)
            mets['wcov'].append(wcov)

    avg_class_ious = class_ious_accumulator / len(test_loader)

    print(f"\n================ FINAL TEST RESULTS ================", flush=True)
    print(f"Semantic -> OA: {np.mean(mets['oa']):.4f} | mAcc: {np.mean(mets['macc']):.4f} | mIoU: {np.mean(mets['miou']):.4f}")
    print(f"Instance -> mPrec: {np.mean(mets['prec']):.4f} | mRec: {np.mean(mets['rec']):.4f} | Cov: {np.mean(mets['cov']):.4f} | WCov: {np.mean(mets['wcov']):.4f}")
    print("--------------------------------------------------")
    for i, name in enumerate(class_names):
        print(f"  {name} Test IoU: {avg_class_ious[i]:.4f}")
    print("==================================================")
                    
if __name__ == '__main__':
    main()


Starting Final Test Evaluation...

================ FINAL TEST RESULTS ================
Semantic -> OA: 0.7927 | mAcc: 0.6911 | mIoU: 0.5151
Instance -> mPrec: 0.2838 | mRec: 0.3628 | Cov: 0.4029 | WCov: 0.6376
--------------------------------------------------
  Railings Test IoU: 0.6953
  Deck Test IoU: 0.7977
  Girder Test IoU: 0.6240
  Cross-girder Test IoU: 0.0368
  Pier cap Test IoU: 0.5521
  Pier Test IoU: 0.7509
  Culvert Test IoU: 0.0000
  Bearings Test IoU: 0.0000


In [21]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
base_dir = r"/storage/jothi/Yash/Datasets/Results_from_133_datasets_with_surfnormals_modified"
test_data_list  = np.load(f"{base_dir}/bridge_test_mod.npy", allow_pickle=True)
test_loader  = DataLoader(BridgeDataset(test_data_list, False), batch_size=1, shuffle=False)
# ==========================================
# FINAL TEST EVALUATION (EXACT LITERATURE METRICS)
# ==========================================
print("\nStarting Final Test Evaluation...", flush=True)
model = ExactJSNet(num_classes=8, emb_dim=5, in_channels=4).cuda()
model.load_state_dict(torch.load(f"{base_dir}/best_optimized_jsnet.pth"))
model.eval()

class_names = ['Railings', 'Deck', 'Girder', 'Cross-girder', 'Pier cap', 'Pier', 'Culvert', 'Bearings']

mets = {
    'oa': [], 'macc': [], 'miou': [],
    'prec': [], 'rec': [], 'cov': [], 'wcov': []
}
class_ious_accumulator = np.zeros(8)

with torch.no_grad():
    for batch in test_loader:
        points = batch['points'].cuda()
        sem_labels = batch['semantic'].cuda()
        inst_labels = batch['instance'].cuda()

        # UPDATED: Split points into spatial coordinates and extra features for inference
        xyz = points[:, :3, :].contiguous()
        features = points[:, 3:, :].contiguous()

        sem_preds, inst_embeds = model(xyz, features)

        # 1. Literature Semantic Evaluation
        oa, macc, miou, class_ious = eval_semantic(sem_preds[0], sem_labels[0], num_classes=8)
        mets['oa'].append(oa)
        mets['macc'].append(macc)
        mets['miou'].append(miou)
        class_ious_accumulator += class_ious

        # 2. Literature Instance Evaluation (MeanShift)
        prec, rec, cov, wcov = eval_instance(inst_embeds[0], inst_labels[0], bandwidth=0.6)
        mets['prec'].append(prec)
        mets['rec'].append(rec)
        mets['cov'].append(cov)
        mets['wcov'].append(wcov)

avg_class_ious = class_ious_accumulator / len(test_loader)

print(f"\n================ FINAL TEST RESULTS ================", flush=True)
print(f"Semantic -> OA: {np.mean(mets['oa']):.4f} | mAcc: {np.mean(mets['macc']):.4f} | mIoU: {np.mean(mets['miou']):.4f}")
print(f"Instance -> mPrec: {np.mean(mets['prec']):.4f} | mRec: {np.mean(mets['rec']):.4f} | Cov: {np.mean(mets['cov']):.4f} | WCov: {np.mean(mets['wcov']):.4f}")
print("--------------------------------------------------")
for i, name in enumerate(class_names):
    print(f"  {name} Test IoU: {avg_class_ious[i]:.4f}")
print("==================================================")


Starting Final Test Evaluation...



================ FINAL TEST RESULTS ================
Semantic -> OA: 0.7939 | mAcc: 0.6920 | mIoU: 0.5159
Instance -> mPrec: 0.2868 | mRec: 0.3463 | Cov: 0.4063 | WCov: 0.6469
--------------------------------------------------
  Railings Test IoU: 0.6994
  Deck Test IoU: 0.7998
  Girder Test IoU: 0.6223
  Cross-girder Test IoU: 0.0358
  Pier cap Test IoU: 0.5482
  Pier Test IoU: 0.7486
  Culvert Test IoU: 0.0000
  Bearings Test IoU: 0.0095


In [23]:
class_names = ['Railings', 'Deck', 'Girder', 'Cross-girder', 'Pier cap', 'Pier', 'Culvert', 'Bearings']
print(f"\n================ FINAL TEST RESULTS ================", flush=True)
print(f"Semantic -> OA: {np.mean(mets['oa']):.4f} | mAcc: {np.mean(mets['macc']):.4f} | mIoU: {np.mean(mets['miou']):.4f}")
print(f"Instance -> mPrec: {np.mean(mets['prec']):.4f} | mRec: {np.mean(mets['rec']):.4f} | Cov: {np.mean(mets['cov']):.4f} | WCov: {np.mean(mets['wcov']):.4f}")
print("--------------------------------------------------")
for i, name in enumerate(class_names):
    print(f"  {name} Test IoU: {avg_class_ious[i]:.4f}")
print("==================================================")


================ FINAL TEST RESULTS ================
Semantic -> OA: 0.7939 | mAcc: 0.6920 | mIoU: 0.5159
Instance -> mPrec: 0.2868 | mRec: 0.3463 | Cov: 0.4063 | WCov: 0.6469
--------------------------------------------------
  Railings Test IoU: 0.6994
  Deck Test IoU: 0.7998
  Girder Test IoU: 0.6223
  Cross-girder Test IoU: 0.0358
  Pier cap Test IoU: 0.5482
  Pier Test IoU: 0.7486
  Culvert Test IoU: 0.0000
  Bearings Test IoU: 0.0095
